# DK cluster demographics — from `data/nodes_DK_pop_DK_clusters.csv`

Reads the **already-computed, AT-consistent** per-cluster aggregates directly — no individual-level
data (`cohort.mat` / `patient_info.pkl` / result pkls) required.

Computed like the Austrian pipeline:
- `size_alive` = person-year observations with `year <= yod` (alive person-years)
- `mortality_rate = #deaths / size_alive` (`#deaths` = distinct final-timestep patients who die); alive-corrected
- `mean_age` = mean of `ages >= 0`, **including** post-death phantom observations

> Danish-**population** files carry `size_alive` + `mortality_rate`; Austrian-population files carry only `size`,
> `mean_age`, `female_ratio` (mortality is Danish-cohort only). Values `<5` are suppressed (blank).


In [ ]:
import os, pandas as pd, numpy as np

# works from the repo root (data/...), a notebooks/ subdir (../data/...), or the local working copy
CANDIDATES = [
    'data/nodes_DK_pop_DK_clusters.csv',
    '../data/nodes_DK_pop_DK_clusters.csv',
    '/Users/einsie0004/Documents/research/X_disease_trajectories/viz/sankey_v2/start/src/data/nodes_DK_pop_DK_clusters.csv',
]
DATA = next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
print('reading:', DATA)
df = pd.read_csv(DATA)
df['cluster'] = df['cluster'].astype(int)
df = df.sort_values('cluster').reset_index(drop=True)
HAS_ALIVE = 'size_alive' in df.columns and 'mortality_rate' in df.columns
print(f'{len(df)} clusters | columns: {list(df.columns)} | size_alive present: {HAS_ALIVE}')
df.head()

In [ ]:
# totals, phantom fraction, cohort self-identification
size = df['size'].sum()
print(f'total obs (size, incl post-death phantoms) = {size:,}')
if HAS_ALIVE:
    alive = df['size_alive'].sum()
    print(f'alive obs (size_alive, year<=yod)          = {alive:,}')
    print(f'post-death phantom obs                     = {size-alive:,}  ({(size-alive)/size:.2%} of total)')
for country, obs in [('Denmark', 42_093_844), ('Austria', 66_466_543)]:
    if abs(size - obs) <= max(5000, 0.001*obs):
        print(f'-> total size matches {country} (paper Table 1 Observations = {obs:,})')

In [ ]:
# reconstruct #deaths and phantom fraction (NaN-safe: mortality is blank where deaths<5)
if HAS_ALIVE:
    df['deaths']       = (df['mortality_rate'] * df['size_alive']).round().astype('Int64')
    df['phantom_obs']  = df['size'] - df['size_alive']
    df['phantom_frac'] = df['phantom_obs'] / df['size']
    print('reconstructed deaths (excl. suppressed clusters):', int(df['deaths'].sum()))
    print('clusters with suppressed (<5) mortality:', int(df['mortality_rate'].isna().sum()))
else:
    print('Austrian-population file: no size_alive / mortality_rate (Danish-cohort only).')

### Highest-mortality clusters


In [ ]:
cols = [c for c in ['cluster','size','size_alive','deaths','phantom_frac','mortality_rate','mean_age','female_ratio'] if c in df.columns]
(df.sort_values('mortality_rate', ascending=False) if HAS_ALIVE else df)[cols].head(10)

### Largest post-death phantom fraction (most alive-correction)


In [ ]:
df.sort_values('phantom_frac', ascending=False)[cols].head(10) if HAS_ALIVE else df[cols].head(10)

### Notes / limitations
- `deaths`, `phantom_obs` are reconstructed from published columns via the AT-method identities
  (`deaths = mortality_rate x size_alive`, `phantom_obs = size - size_alive`).
- `mortality_rate` is **alive-corrected** (`#deaths / size_alive`), matching the Austrian pipeline. It therefore
  differs from the crude `#deaths / size` figure reported in the paper for high-mortality clusters.
- The **alive-corrected mean age** cannot be recomputed here (needs per-observation `ages`/`is_alive`); `mean_age`
  is the phantom-inclusive value. Set `DATA` to another `nodes_*_pop_*_clusters.csv` to inspect other quadrants.
